# CryptoPulse Sesión 2

Carga histórica, transformaciones, funciones, agrupaciones, evaluación perezosa y RDD con BTCUSDT y SOLUSDT. El notebook usa exclusivamente los archivos reales registrados en `data/provenance.json`.

El intervalo inicial es diciembre de 2024 a enero de 2025. No representa todavía todo el histórico previsto desde 2022. Las variables calculadas son descriptivas y usan información disponible al cierre de cada vela; aquí no se entrenan modelos.


In [1]:
import os
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--driver-memory 2g pyspark-shell")

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import math
import contextlib
from decimal import Decimal

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DecimalType, StringType
)
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.appName("cryptopulse-s02-fundamentos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
SESSION = Path("/opt/s02-fundamentos")
DATA = SESSION / "data"
ARTIFACTS = SESSION / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
# El resultado PASS se crea al finalizar esta corrida.
(ARTIFACTS / "verificacion_s02.json").unlink(missing_ok=True)
print("Spark:", spark.version, "| UTC | master:", spark.sparkContext.master)
print("Heap máximo JVM MiB:", spark._jvm.java.lang.Runtime.getRuntime().maxMemory() // (1024 * 1024))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 22:11:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/26 22:12:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 4.2.0 | UTC | master: local[*]
Heap máximo JVM MiB: 2048


## Procedencia e integridad

Los ZIP se verifican contra el SHA256 oficial durante la descarga. Antes de leerlos con Spark se verifica también el hash de cada CSV extraído. Una ampliación del rango actualiza el manifiesto; este notebook lee solo los archivos enumerados allí.


In [2]:
manifest = json.loads((DATA / "provenance.json").read_text(encoding="utf-8"))
assert manifest["synthetic"] is False
csv_paths = []
for item in manifest["archives"]:
    assert item["checksum_verified"]
    path = DATA / item["csv_relative_path"]
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    assert digest.hexdigest() == item["sha256_csv"], f"CSV alterado: {path.name}"
    csv_paths.append(str(path))
print("Intervalo mensual:", manifest["month_from"], "a", manifest["month_to"])
print("CSV verificados:", len(csv_paths), "| filas previstas:", manifest["rows_total"])


Intervalo mensual: 2024-12 a 2025-01
CSV verificados: 4 | filas previstas: 178560


## Lectura y exploración

Primero se muestra la inferencia sobre un archivo. Después se aplica un esquema explícito a todo el intervalo. Los CSV originales no tienen cabecera y contienen 12 campos. Los precios y volúmenes se conservan como decimales; la inferencia puede elegir otros tipos.


In [3]:
df_inferido = spark.read.option("header", False).option("inferSchema", True).csv(csv_paths[0])
df_inferido.printSchema()
df_inferido.show(2, truncate=False)

schema_raw = StructType([
    StructField("open_time_raw", LongType()),
    StructField("open", DecimalType(28, 8)),
    StructField("high", DecimalType(28, 8)),
    StructField("low", DecimalType(28, 8)),
    StructField("close", DecimalType(28, 8)),
    StructField("volume", DecimalType(28, 8)),
    StructField("close_time_raw", LongType()),
    StructField("quote_asset_volume", DecimalType(28, 8)),
    StructField("number_of_trades", LongType()),
    StructField("taker_buy_base_volume", DecimalType(28, 8)),
    StructField("taker_buy_quote_volume", DecimalType(28, 8)),
    StructField("ignore", StringType()),
])
df_raw = (
    spark.read.schema(schema_raw).option("header", False)
    .option("mode", "FAILFAST").csv(csv_paths)
    .withColumn("source_file", F.regexp_extract(F.input_file_name(), r"([^/]+)$", 1))
    .withColumn("symbol", F.regexp_extract("source_file", r"^(BTCUSDT|SOLUSDT)-1m-", 1))
    .withColumn("archive_month", F.regexp_extract("source_file", r"-(\d{4}-\d{2})\.csv$", 1))
)
df_raw.printSchema()


root
 |-- _c0: long (nullable = true)
 |-- _c1: double (nullable = true)
 |-- _c2: double (nullable = true)
 |-- _c3: double (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: double (nullable = true)
 |-- _c6: long (nullable = true)
 |-- _c7: double (nullable = true)
 |-- _c8: integer (nullable = true)
 |-- _c9: double (nullable = true)
 |-- _c10: double (nullable = true)
 |-- _c11: integer (nullable = true)

+-------------+--------+--------+--------+--------+-------+-------------+--------------+----+-------+--------------+----+
|_c0          |_c1     |_c2     |_c3     |_c4     |_c5    |_c6          |_c7           |_c8 |_c9    |_c10          |_c11|
+-------------+--------+--------+--------+--------+-------+-------------+--------------+----+-------+--------------+----+
|1733011200000|96407.99|96411.99|96376.0 |96376.0 |8.18452|1733011259999|788996.1628126|1763|3.18577|307128.8909746|0   |
|1733011260000|96376.0 |96384.76|96350.13|96350.13|6.72047|1733011319999|647671.036216

## Transformaciones y evaluación perezosa

Construir un filtro y una selección no ejecuta filas. La acción `count()` sí crea trabajos Spark. El plan muestra las columnas solicitadas y los filtros enviados al lector. El detalle de la optimización depende de la versión y del origen.


In [4]:
sc = spark.sparkContext
lazy_group = "cryptopulse-s02-lazy"
sc.setJobGroup(lazy_group, "Comparar transformaciones con acciones")
jobs_before = set(sc.statusTracker().getJobIdsForGroup(lazy_group))

seleccion = (
    df_raw.select("symbol", "open_time_raw", "close", "volume")
    .filter((F.col("symbol") == "BTCUSDT") & (F.col("volume") > 0))
)
jobs_after_transform = set(sc.statusTracker().getJobIdsForGroup(lazy_group))
assert jobs_after_transform == jobs_before

plan_buffer = io.StringIO()
with contextlib.redirect_stdout(plan_buffer):
    seleccion.explain(extended=True)
plan_text = plan_buffer.getvalue()
print(plan_text)
(ARTIFACTS / "plan_ejecucion.txt").write_text(plan_text, encoding="utf-8")

selected_count = seleccion.count()
jobs_after_action = set(sc.statusTracker().getJobIdsForGroup(lazy_group))
assert jobs_after_action - jobs_after_transform, "count() no registro un job"
lazy_check = {
    "new_jobs_after_transformations": len(jobs_after_transform - jobs_before),
    "new_jobs_after_action": len(jobs_after_action - jobs_after_transform),
    "selected_btc_rows_positive_volume": selected_count,
}
print(json.dumps(lazy_check, indent=2))
sc.setJobGroup("cryptopulse-s02-analisis", "Normalizacion y analisis BTC/SOL")


== Parsed Logical Plan ==
'Filter 'and('`=`('symbol, BTCUSDT), '`>`('volume, 0))
+- Project [symbol#93, open_time_raw#79L, close#83, volume#84]
   +- Project [open_time_raw#79L, open#80, high#81, low#82, close#83, volume#84, close_time_raw#85L, quote_asset_volume#86, number_of_trades#87L, taker_buy_base_volume#88, taker_buy_quote_volume#89, ignore#90, source_file#92, symbol#93, regexp_extract(source_file#92, -(\d{4}-\d{2})\.csv$, 1) AS archive_month#94]
      +- Project [open_time_raw#79L, open#80, high#81, low#82, close#83, volume#84, close_time_raw#85L, quote_asset_volume#86, number_of_trades#87L, taker_buy_base_volume#88, taker_buy_quote_volume#89, ignore#90, source_file#92, regexp_extract(source_file#92, ^(BTCUSDT|SOLUSDT)-1m-, 1) AS symbol#93]
         +- Project [open_time_raw#79L, open#80, high#81, low#82, close#83, volume#84, close_time_raw#85L, quote_asset_volume#86, number_of_trades#87L, taker_buy_base_volume#88, taker_buy_quote_volume#89, ignore#90, regexp_extract(input_file

## Normalización UTC y columnas derivadas

Binance Spot usa milisegundos antes de 2025 y microsegundos desde 2025. Se utiliza el mes del archivo para convertir ambas escalas. `event_time` es el cierre de vela, cuando sus valores completos están disponibles.

El rango y el cuerpo porcentual describen la propia vela; no son umbrales ni alertas de riesgo. El volumen base se expresa en BTC o SOL; el volumen cotizado, en USDT.


In [5]:
df = (
    df_raw.withColumn("timestamp_unit", F.when(F.col("archive_month") >= "2025-01", F.lit("us")).otherwise(F.lit("ms")))
    .withColumn("open_time", F.when(F.col("timestamp_unit") == "us",
        F.timestamp_micros("open_time_raw")).otherwise(F.timestamp_millis("open_time_raw")))
    .withColumn("close_time", F.when(F.col("timestamp_unit") == "us",
        F.timestamp_micros("close_time_raw")).otherwise(F.timestamp_millis("close_time_raw")))
    .withColumn("event_time", F.col("close_time"))
    .withColumn("date_utc", F.to_date("open_time"))
    .withColumn("hour_utc", F.hour("open_time"))
    .withColumn("day_of_week", F.dayofweek("open_time"))
    .withColumn("day_type", F.when(F.col("day_of_week").isin(1, 7), "fin_de_semana").otherwise("laborable"))
    .withColumn("base_asset", F.regexp_replace("symbol", "USDT$", ""))
    .withColumn("is_closed", F.lit(True))
    .withColumn("source", F.lit("binance_public_data_spot"))
    .withColumn("range_pct", 100.0 * (F.col("high").cast("double") - F.col("low").cast("double")) / F.col("open").cast("double"))
    .withColumn("body_pct", 100.0 * (F.col("close").cast("double") - F.col("open").cast("double")) / F.col("open").cast("double"))
    .withColumn("candle_direction", F.when(F.col("close") > F.col("open"), "sube")
        .when(F.col("close") < F.col("open"), "baja").otherwise("sin_cambio"))
    .drop("ignore")
)
df.select("symbol", "open_time", "close_time", "close", "volume", "quote_asset_volume",
          "timestamp_unit", "range_pct", "candle_direction").orderBy("open_time", "symbol").show(6, truncate=False)
df.groupBy("symbol", "timestamp_unit").count().orderBy("symbol", "timestamp_unit").show()
# Estadisticas por activo para evitar mezclar precios y volumen base BTC con SOL.
df.groupBy("symbol").agg(
    F.count("*").alias("velas"),
    F.min("open_time").alias("inicio_utc"), F.max("open_time").alias("ultimo_inicio_utc"),
    F.min("close").alias("cierre_min_usdt"), F.max("close").alias("cierre_max_usdt"),
    F.avg("range_pct").alias("rango_medio_pct")
).orderBy("symbol").show(truncate=False)


+-------+-------------------+-----------------------+--------------+-------------+------------------+--------------+--------------------+----------------+
|symbol |open_time          |close_time             |close         |volume       |quote_asset_volume|timestamp_unit|range_pct           |candle_direction|
+-------+-------------------+-----------------------+--------------+-------------+------------------+--------------+--------------------+----------------+
|BTCUSDT|2024-12-01 00:00:00|2024-12-01 00:00:59.999|96376.00000000|8.18452000   |788996.16281260   |ms            |0.037330930766220974|baja            |
|SOLUSDT|2024-12-01 00:00:00|2024-12-01 00:00:59.999|237.26000000  |7569.52500000|1796207.36433000  |ms            |0.2525252525252621  |baja            |
|BTCUSDT|2024-12-01 00:01:00|2024-12-01 00:01:59.999|96350.13000000|6.72047000   |647671.03621690   |ms            |0.03593218228603605 |baja            |
|SOLUSDT|2024-12-01 00:01:00|2024-12-01 00:01:59.999|237.26000000  |23

## Ventanas y rendimientos

`lag()` toma el cierre anterior del mismo activo. Se calcula el rendimiento de un minuto solo cuando los inicios de vela están separados exactamente 60 segundos. Un hueco no se convierte en un retorno de un minuto.

La volatilidad realizada de los últimos 15 minutos se define aquí como `100 × raíz(sumatoria de retornos logarítmicos al cuadrado)`, sin anualizar. Requiere 15 retornos válidos consecutivos, por lo tanto 16 cierres. No contiene información futura.

Los precios originales permanecen en Decimal. Los cocientes usados en rendimientos se calculan en doble precisión para evitar que las reglas de escala de división Decimal redondeen el cambio antes de aplicar el logaritmo.


In [ ]:
order_window = Window.partitionBy("symbol").orderBy("open_time")
past_15 = order_window.rowsBetween(-14, 0)
features = (
    df.withColumn("previous_close", F.lag("close").over(order_window))
    .withColumn("previous_open_time", F.lag("open_time").over(order_window))
    .withColumn("seconds_since_previous", F.col("open_time").cast("long") - F.col("previous_open_time").cast("long"))
    .withColumn("return_1m_pct", F.when(
        (F.col("seconds_since_previous") == 60) & (F.col("previous_close") > 0),
        100.0 * (F.col("close").cast("double") / F.col("previous_close").cast("double") - 1)))
    .withColumn("log_return_1m", F.when(
        (F.col("seconds_since_previous") == 60) & (F.col("previous_close") > 0) & (F.col("close") > 0),
        F.log(F.col("close").cast("double") / F.col("previous_close").cast("double"))))
    .withColumn("valid_returns_15m", F.count("log_return_1m").over(past_15))
    .withColumn("realized_volatility_15m_pct", F.when(
        F.col("valid_returns_15m") == 15,
        100.0 * F.sqrt(F.sum(F.pow("log_return_1m", 2)).over(past_15))))
    .withColumn("rolling_quote_volume_15m_usdt", F.when(
        (F.count(F.lit(1)).over(past_15) == 15)
        & ((F.col("open_time").cast("long") - F.min("open_time").over(past_15).cast("long")) == 14 * 60),
        F.sum("quote_asset_volume").over(past_15)))
    .cache()
)
total_rows = features.count()
features.select("symbol", "open_time", "close", "return_1m_pct",
                "realized_volatility_15m_pct").orderBy("open_time", "symbol").show(36, truncate=False)


## Agrupaciones y agregaciones

Los resúmenes mantienen `symbol` como clave. Las velas horarias se forman con apertura y cierre según tiempo; no con `first()` sin orden. El VWAP horario divide el importe negociado en USDT entre el volumen base negociado.


In [ ]:
daily = (
    features.groupBy("symbol", "date_utc").agg(
        F.count("*").alias("candles"),
        F.countDistinct("open_time").alias("distinct_minutes"),
        F.sum("volume").alias("volume_base"),
        F.sum("quote_asset_volume").alias("quote_volume_usdt"),
        F.sum("number_of_trades").alias("trades"),
        F.avg("range_pct").alias("avg_range_pct"),
    ).orderBy("date_utc", "symbol")
)
hourly = (
    features.withColumn("hour_start", F.date_trunc("hour", "open_time"))
    .groupBy("symbol", "hour_start").agg(
        F.min_by("open", "open_time").alias("open"),
        F.max("high").alias("high"), F.min("low").alias("low"),
        F.max_by("close", "open_time").alias("close"),
        F.sum("volume").alias("volume_base"),
        F.sum("quote_asset_volume").alias("quote_volume_usdt"),
        F.sum("number_of_trades").alias("trades"),
        F.count("*").alias("candles")
    ).withColumn("vwap_usdt", F.when(F.col("volume_base") > 0,
                                  F.col("quote_volume_usdt") / F.col("volume_base")))
)
daily.show(8, truncate=False)
hourly.orderBy("hour_start", "symbol").show(6, truncate=False)
# Una ventana agrega un resumen sin colapsar las velas originales.
window_day = Window.partitionBy("symbol", "date_utc")
with_day_total = features.withColumn(
    "quote_volume_day_usdt", F.sum("quote_asset_volume").over(window_day)
)
with_day_total.select("symbol", "open_time", "quote_asset_volume", "quote_volume_day_usdt").orderBy(
    "open_time", "symbol").show(4, truncate=False)


## CSV y Parquet

Se guarda una muestra aleatoria reproducible para explorar la mecánica CSV/Parquet. Esta muestra no se utilizará para separar entrenamiento y prueba: las series predictivas necesitarán particiones cronológicas.

Los históricos transformados se guardan completos para reutilizarlos en S3. Todavía no constituyen el dataset final con controles de calidad y particionamiento de esa sesión.


In [ ]:
sample = features.select(
    "symbol", "open_time", "close", "volume", "quote_asset_volume", "number_of_trades"
).sample(withReplacement=False, fraction=0.01, seed=26).cache()
sample_count = sample.count()
timestamp_format = "yyyy-MM-dd'T'HH:mm:ss.SSSSSS'Z'"
sample.write.mode("overwrite").option("header", True).option("timestampFormat", timestamp_format).csv(
    str(ARTIFACTS / "muestra_csv"))
sample.write.mode("overwrite").parquet(str(ARTIFACTS / "muestra_parquet"))
sample_csv = spark.read.schema(sample.schema).option("header", True).option(
    "timestampFormat", timestamp_format).csv(str(ARTIFACTS / "muestra_csv"))
sample_parquet = spark.read.parquet(str(ARTIFACTS / "muestra_parquet"))
assert sample_count > 0
assert sample_csv.count() == sample_count == sample_parquet.count()
assert sample.exceptAll(sample_csv).limit(1).count() == 0
assert sample.exceptAll(sample_parquet).limit(1).count() == 0
features.write.mode("overwrite").parquet(str(ARTIFACTS / "klines_transformadas"))
daily.write.mode("overwrite").parquet(str(ARTIFACTS / "resumen_diario"))
hourly.write.mode("overwrite").parquet(str(ARTIFACTS / "resumen_horario"))
print("Muestra CSV/Parquet:", sample_count, "| historico transformado:", total_rows)


## RDD con datos del proyecto

Se convierten las velas a RDD y se expanden a pares de métricas para contar velas y operaciones por activo con `filter`, `map`, `flatMap` y `reduceByKey`. Se contrasta con las mismas métricas en DataFrame. El procesamiento principal conserva DataFrames para aprovechar Catalyst.


In [ ]:
from operator import add
rdd_rows = features.select("symbol", "number_of_trades").rdd.filter(
    lambda row: row.symbol is not None and row.number_of_trades is not None
)
rdd_values = rdd_rows.map(lambda row: (row.symbol, row.number_of_trades))
rdd_metrics = dict(rdd_values.flatMap(
    lambda item: [((item[0], "candles"), 1), ((item[0], "trades"), item[1])]
).reduceByKey(add).collect())
df_metrics = features.groupBy("symbol").agg(
    F.count("*").alias("candles"), F.sum("number_of_trades").alias("trades")
).collect()
for row in df_metrics:
    assert rdd_metrics[(row.symbol, "candles")] == row.candles
    assert rdd_metrics[(row.symbol, "trades")] == row.trades
print({f"{symbol}.{metric}": value for (symbol, metric), value in sorted(rdd_metrics.items())})


## Comprobación final

Se comprueban el tamaño y el rango leídos, las conversiones UTC, una vela horaria contra 60 filas originales y los retornos alrededor del cambio de escala temporal de 2025. Los huecos se registran; no se inventan velas para rellenarlos.


In [ ]:
expected_counts = {}
for entry in manifest["archives"]:
    expected_counts[entry["symbol"]] = expected_counts.get(entry["symbol"], 0) + entry["rows"]
actual_counts = {row.symbol: row["count"] for row in features.groupBy("symbol").count().collect()}
assert actual_counts == expected_counts
assert total_rows == manifest["rows_total"]
assert features.filter(
    F.col("open_time").isNull() | F.col("close_time").isNull() | F.col("close").isNull()
    | (F.col("symbol") == "") | (F.date_format("open_time", "yyyy-MM") != F.col("archive_month"))
    | (F.col("event_time") != F.col("close_time"))
).count() == 0
assert features.select("symbol", "open_time").distinct().count() == total_rows
gaps = features.filter(
    F.col("previous_open_time").isNotNull() & (F.col("seconds_since_previous") != 60)
).count()
assert features.filter(
    (F.col("seconds_since_previous") != 60) & F.col("return_1m_pct").isNotNull()
).count() == 0

# Referencia independiente: primera hora de un archivo original.
entry = manifest["archives"][0]
with (DATA / entry["csv_relative_path"]).open(newline="", encoding="utf-8") as file:
    original_rows = [next(csv.reader(file)) for _ in range(60)]
scale = 1_000_000 if entry["timestamp_unit"] == "us" else 1_000
first_time = datetime.fromtimestamp(int(original_rows[0][0]) / scale, timezone.utc)
reference_hour = first_time.replace(minute=0, second=0, microsecond=0)
check_hour = hourly.filter(
    (F.col("symbol") == entry["symbol"]) & (F.col("hour_start") == F.lit(reference_hour))
).first()
assert check_hour.candles == 60
assert check_hour.open == Decimal(original_rows[0][1])
assert check_hour.close == Decimal(original_rows[-1][4])
assert check_hour.high == max(Decimal(row[2]) for row in original_rows)
assert check_hour.low == min(Decimal(row[3]) for row in original_rows)
assert check_hour.volume_base == sum(Decimal(row[5]) for row in original_rows)
assert check_hour.quote_volume_usdt == sum(Decimal(row[7]) for row in original_rows)
assert check_hour.trades == sum(int(row[8]) for row in original_rows)

# Referencia independiente para calentamiento y ventanas retrospectivas.
first_minutes = features.filter(F.col("symbol") == entry["symbol"]).orderBy("open_time").limit(16).select(
    "rolling_quote_volume_15m_usdt", "realized_volatility_15m_pct"
).collect()
assert all(row.rolling_quote_volume_15m_usdt is None for row in first_minutes[:14])
assert first_minutes[14].rolling_quote_volume_15m_usdt == sum(Decimal(row[7]) for row in original_rows[:15])
assert all(row.realized_volatility_15m_pct is None for row in first_minutes[:15])
reference_returns = [math.log(float(Decimal(original_rows[i][4]) / Decimal(original_rows[i-1][4])))
                     for i in range(1, 16)]
reference_volatility = 100 * math.sqrt(sum(value * value for value in reference_returns))
assert math.isclose(first_minutes[15].realized_volatility_15m_pct,
                    reference_volatility, rel_tol=1e-8, abs_tol=1e-10), (first_minutes[15].realized_volatility_15m_pct, reference_volatility)

# El rango inicial cruza el cambio ms/us. Otros rangos pueden no cruzarlo.
units = sorted({entry["timestamp_unit"] for entry in manifest["archives"]})
boundary_checks = 0
if manifest["month_from"] <= "2024-12" and manifest["month_to"] >= "2025-01":
    for symbol in manifest["symbols"]:
        boundary = features.filter(
            (F.col("symbol") == symbol)
            & F.col("open_time").between("2024-12-31 23:59:00", "2025-01-01 00:00:00")
        ).orderBy("open_time").select("close", "log_return_1m", "seconds_since_previous").collect()
        assert len(boundary) == 2 and boundary[1].seconds_since_previous == 60
        expected_return = math.log(float(boundary[1].close / boundary[0].close))
        assert math.isclose(boundary[1].log_return_1m, expected_return, rel_tol=1e-6, abs_tol=1e-12)
        boundary_checks += 1

# Verificar tambien que el Parquet completo es legible.
saved_count = spark.read.parquet(str(ARTIFACTS / "klines_transformadas")).count()
assert saved_count == total_rows
verification = {
    "status": "PASS", "project": "CryptoPulse", "session": "S02",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version, "master": sc.master,
    "application_id": sc.applicationId, "spark_ui_internal": sc.uiWebUrl,
    "month_from": manifest["month_from"], "month_to": manifest["month_to"],
    "source_files": len(csv_paths), "source_hashes_verified": True,
    "rows_total": total_rows, "rows_per_symbol": actual_counts,
    "raw_timestamp_units": units, "ms_us_boundary_checks": boundary_checks,
    "gaps_detected": gaps, "hourly_ohlcv_reference": "PASS",
    "rolling_window_reference": "PASS",
    "lazy_evaluation": lazy_check, "rdd_matches_dataframe": True,
    "csv_parquet_sample_rows": sample_count, "sample_roundtrip": "PASS",
    "transformed_parquet_rows": saved_count,
    "note": "Variables descriptivas; no hay modelos entrenados. Controles formales de calidad y particionado siguen en S03.",
}
(ARTIFACTS / "verificacion_s02.json").write_text(
    json.dumps(verification, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(verification, ensure_ascii=False, indent=2))
sample.unpersist()
features.unpersist()


Con `status: PASS`, S2 queda comprobada sobre el intervalo descargado. La próxima sesión aplicará controles formales de calidad y almacenará los datos en Parquet particionado. El notebook conserva todos los resultados en la copia ejecutada de `artifacts/`.
